In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação térmica (RandomForestRegressor) aprendida apenas com dados sem falha
+ Aplicação global + Classificação multiclasse (0=sem, 1=falha1, 2=falha2)
+ Split por temperatura (temperaturas no teste não aparecem no treino)
+ Tempo total de execução de cada parte
Autor: Luiz Eduardo Abdala José
"""

import re, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# ===================== PARÂMETROS =====================
ARQ_BASE = "base-completo--.pkl"  # base com 3 classes
REF_TEMP = 20
FREQ_MIN_KHZ = 30
FREQ_MAX_KHZ = 45
SMOOTH_WIN = 5
TAU_MAX_FRAC = 0.025
ANCHOR_TO_REF_ENDS = True
CAPS = dict(gain_frac=0.60, offset_frac=0.60, tilt_frac=0.40)

# ===================== FUNÇÕES AUXILIARES =====================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f / 1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def moving_average(arr, win):
    if win <= 1 or win % 2 == 0: return arr
    r = win // 2
    padl = np.repeat(arr[:1], r)
    padr = np.repeat(arr[-1:], r)
    x = np.concatenate([padl, arr, padr])
    c = np.cumsum(x, dtype=float)
    c = np.concatenate([[0.0], c])
    s = c[win:] - c[:-win]
    return s / float(win)

def shift_interp(x_row, fhz, tau_hz):
    f_shift = fhz + float(tau_hz)
    return np.interp(fhz, f_shift, x_row, left=x_row[0], right=x_row[-1])

def energy_weighted_centroid(f, x):
    xm = np.asarray(x, float)
    w = xm * xm
    den = float(np.trapezoid(w, f))
    if den <= 1e-18:
        return float(np.mean(f))
    num = float(np.trapezoid(f * w, f))
    return num / den

def slope_over_band(f, x):
    return float((x[-1] - x[0]) / (f[-1] - f[0] + 1e-12))

# ===== FEATURES =====
def compute_features(X, f):
    X = np.asarray(X, float); n, m = X.shape
    out = []
    for i in range(n):
        x = X[i]
        mean = float(np.mean(x))
        std = float(np.std(x))
        amp = float(x.max() - x.min())
        slope = slope_over_band(f, x)
        centroid = energy_weighted_centroid(f, x)
        out.append([mean, std, amp, slope, centroid])
    cols = ["mean", "std", "amp", "slope", "centroid"]
    return np.array(out, float), cols

# ===== MODELAGEM RANDOM FOREST (feature × temperatura) =====
def fit_feature_vs_temp_models(F, T):
    models = {}
    T = np.asarray(T, float).reshape(-1, 1)
    for j, name in enumerate(["mean", "std", "amp", "slope", "centroid"]):
        rf = RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_split=4,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(T, F[:, j])
        models[name] = rf
    return models

def feature_targets_at_ref(models, ref_temp=REF_TEMP):
    Tref = np.array([[ref_temp]])
    return {name: float(m.predict(Tref)[0]) for name, m in models.items()}

# ===== COMPENSAÇÃO =====
def apply_compensation_by_features(x, f, targets, caps, y_ref=None):
    x = x.copy()
    mean_t = targets["mean"]; amp_t = targets["amp"]; slope_t = targets["slope"]
    centroid_t = targets["centroid"]

    mean_x = float(x.mean()); amp_x = float(x.max() - x.min()); slope_x = slope_over_band(f, x)

    # offset
    offset = mean_t - mean_x
    offset_cap = caps["offset_frac"] * max(1e-9, amp_x)
    offset = float(np.clip(offset, -offset_cap, offset_cap))
    x = x + offset

    # ganho
    gain = 1.0 if amp_x <= 1e-9 else float(amp_t / amp_x)
    gmin = 1.0 - caps["gain_frac"]; gmax = 1.0 + caps["gain_frac"]
    gain = float(np.clip(gain, gmin, gmax))
    x = mean_t + gain * (x - mean_t)

    # tilt
    delta_slope = slope_t - slope_x
    u = np.linspace(-0.5, 0.5, len(x))
    df = (f[-1] - f[0] + 1e-12)
    tilt_signal = (delta_slope * df) * u
    tilt_cap = caps["tilt_frac"] * max(1e-9, amp_x)
    tilt_signal = np.clip(tilt_signal, -tilt_cap, tilt_cap)
    x = x + tilt_signal

    # shift (centroide)
    cent_x = energy_weighted_centroid(f, x)
    delta_c = centroid_t - cent_x
    tau_max = TAU_MAX_FRAC * (f[-1] - f[0])
    tau = float(np.clip(delta_c, -tau_max, tau_max))
    if abs(tau) > 1e-12:
        x = shift_interp(x, f, tau)

    # ancoragem nos extremos
    if ANCHOR_TO_REF_ENDS and (y_ref is not None):
        e0 = x[0] - y_ref[0]; e1 = x[-1] - y_ref[-1]
        corr = np.linspace(e0, e1, len(x))
        x = x - corr

    return x

def compensate_all(df, fcols, fhz, ref_temp=REF_TEMP):
    print("🔹 Treinando modelo de compensação (somente sem falha)...")
    t0 = time.time()
    df_sem = df[df["falha"] == 0].copy()
    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)
    F_sem, _ = compute_features(X_sem, fhz)
    feat_models = fit_feature_vs_temp_models(F_sem, T_sem)
    targets = feature_targets_at_ref(feat_models, ref_temp)

    # referência sem falha a 20°C
    pool_ref = df_sem.loc[np.isclose(df_sem["temperatura_c"], ref_temp), fcols].to_numpy(float)
    if len(pool_ref) == 0:
        print("⚠️ Nenhum dado exato a 20°C sem falha encontrado, usando média global sem falha.")
        y_ref = np.median(X_sem, axis=0)
    else:
        y_ref = np.median(pool_ref, axis=0)
    print(f"🔹 Referência calculada a partir de {len(pool_ref)} amostras sem falha a {REF_TEMP}°C.")

    print("🔹 Aplicando compensação em toda a base...")
    X_all = df[fcols].to_numpy(float)
    Y = np.zeros_like(X_all)
    for i in range(X_all.shape[0]):
        y = apply_compensation_by_features(X_all[i], fhz, targets, CAPS, y_ref=y_ref)
        if SMOOTH_WIN > 1 and SMOOTH_WIN % 2 == 1:
            y = moving_average(y, SMOOTH_WIN)
        Y[i] = y
    df_comp = df.copy()
    df_comp[fcols] = Y
    print(f"✅ Compensação concluída em {time.time()-t0:.2f} s.")
    return df_comp, y_ref

# ===================== EXECUÇÃO PRINCIPAL =====================
print("🔹 Carregando base de dados...")
t_total = time.time()
df = pd.read_pickle(ARQ_BASE)
fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
df_comp, y_ref = compensate_all(df, fcols, fhz, REF_TEMP)

# ===================== CLASSIFICAÇÃO (split por temperatura) =====================
print("\n🔹 Dividindo treino/teste por temperatura...")
temps_all = sorted(df_comp["temperatura_c"].unique())
n_train = int(len(temps_all) * 0.7)
np.random.seed(42)
temps_train = set(np.random.choice(temps_all, n_train, replace=False))
temps_test = set(t for t in temps_all if t not in temps_train)
print(f"Temperaturas treino: {sorted(temps_train)}")
print(f"Temperaturas teste:  {sorted(temps_test)}")

df_train = df_comp[df_comp["temperatura_c"].isin(temps_train)].copy()
df_test = df_comp[df_comp["temperatura_c"].isin(temps_test)].copy()

X_train = df_train[fcols].to_numpy(float)
y_train = df_train["falha"].to_numpy(int)
X_test = df_test[fcols].to_numpy(float)
y_test = df_test["falha"].to_numpy(int)

print(f"Amostras treino: {len(X_train)} | teste: {len(X_test)}")

print("\n🔹 Treinando RandomForestClassifier (multiclasse)...")
t0 = time.time()
clf = RandomForestClassifier(
    n_estimators=400, max_depth=10,
    min_samples_split=4, min_samples_leaf=3,
    random_state=42, n_jobs=-1
)
clf.fit(X_train, y_train)
train_time = time.time() - t0
print(f"✅ Classificador treinado em {train_time:.2f} s")

t0 = time.time()
y_pred = clf.predict(X_test)
pred_time = time.time() - t0
print(f"⏱️ Tempo de predição: {pred_time:.3f} s")

print("\n== RESULTADOS RANDOM FOREST (após compensação, 3 classes, split por temperatura) ==")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

# ===================== PLOT DE EXEMPLO =====================
print("\n🔹 Gerando gráfico de exemplo...")
fhz_khz = fhz / 1e3
idx_show = df_test.index[10]  # primeira amostra do teste
plt.figure(figsize=(9, 5))
plt.plot(fhz_khz, y_ref, '--', c='black', lw=1.2, label=f"Referência {REF_TEMP}°C (sem falha)")
plt.plot(fhz_khz, df.loc[idx_show, fcols], c='tab:red', alpha=0.6,
         label=f"Original {df.loc[idx_show,'temperatura_c']}°C (falha={df.loc[idx_show,'falha']})")
plt.plot(fhz_khz, df_comp.loc[idx_show, fcols], c='tab:blue', lw=2,
         label=f"Compensado {df.loc[idx_show,'temperatura_c']}°C")
plt.title(f"Compensação RF — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real da impedância")
plt.legend()
plt.tight_layout()
plt.show()

print(f"\n✅ Execução total concluída em {time.time()-t_total:.2f} s.")


In [ ]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.spatial.distance import cosine
from math import acos, degrees

def calc_metrics(y_true, y_pred):
    """Calcula todas as métricas entre curvas"""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    corr = np.corrcoef(y_true, y_pred)[0,1]
    # SAM (Spectral Angle Mapper)
    sam_rad = acos(np.clip(np.dot(y_true, y_pred) /
                           (np.linalg.norm(y_true) * np.linalg.norm(y_pred) + 1e-12), -1, 1))
    sam_deg = degrees(sam_rad)
    nrmse = rmse / (y_true.max() - y_true.min() + 1e-12)
    rmsd = np.sqrt(np.mean((y_true - y_pred - np.mean(y_true - y_pred))**2))
    ccdm = 1 - corr
    return dict(R2=r2, RMSE=rmse, MAE=mae, Corr=corr,
                SAM_deg=sam_deg, NRMSE=nrmse, RMSD=rmsd, CCDM=ccdm)

# ===== Exemplo de uso =====
# y_ref: referência 20°C (ex: y_ref do código principal)
# X_orig: curva original (ex: df.loc[idx_show, fcols])
# X_comp: curva compensada (ex: df_comp.loc[idx_show, fcols])

y_ref_vec = y_ref
X_orig = df.loc[idx_show, fcols].to_numpy(float)
X_comp = df_comp.loc[idx_show, fcols].to_numpy(float)

print("\n== MÉTRICAS ORIGINAL vs REFERÊNCIA ==")
print(calc_metrics(y_ref_vec, X_orig))

print("\n== MÉTRICAS COMPENSADO vs REFERÊNCIA ==")
print(calc_metrics(y_ref_vec, X_comp))
